# Collection A Processing: Split by Method (TMT, iTRAQ, Label-Free)

This notebook processes **Collection A** cohorts by grouping them into 3 distinct categories based on experimental method:
1.  **TMT** (CPTAC/ICPC/CNHPP)
2.  **iTRAQ** (Historical CPTAC)
3.  **Label-Free** (Other matched cohorts)

For each group, we:
- Load the cohort datasets.
- Concatenate them into a unified matrix.
- Filter Features (>50% missing) and Samples (>50% missing).
- Save separate integrated files for Protein and RNA.

In [1]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "1. process_collection_a"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

Figures are saved to C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\05. validation\notebooks\figures_pdf\1. process_collection_a


In [2]:
from pathlib import Path
import sys

# Make the adjacent src/ package importable when this notebook is run from the notebooks folder.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import src.utils_val as uv

In [3]:
import os
import glob
import pandas as pd
from tqdm import tqdm
import warnings


In [4]:

warnings.simplefilter(action='ignore', category=FutureWarning)

DATA_DIR = r"../data/Collection_A"
OUTPUT_DIR = "../data/processed"
if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

# --- DEFINING COHORT GROUPS ---
COHORT_GROUPS = {
    "TMT": [
        # CPTAC / ICPC / CNHPP
        "colon_2019", "renal_2019", "Endomet_val", "breast2020", "pedbrain_2020",
        "lung_2020_smoke", "lung_val", "lung2021", "headneck_2021", "pancrea2021",
        "brain_2021", "liver_val"
    ],
    "iTRAQ": [
        # Historical CPTAC
        "breast_2016", "ovary_2016", "stomach_2019"
    ],
    "LabelFree": [
        # Other matched cohorts
        "brain", "colon", "prostate", "liver", "lung"
    ]
}

# Initialize storage
group_data = {"TMT": {"prot": [], "rna": []}, "iTRAQ": {"prot": [], "rna": []}, "LabelFree": {"prot": [], "rna": []}}


## 1. Load Data into Groups

In [5]:
subdirs = [f.path for f in os.scandir(DATA_DIR) if f.is_dir()]
print(f"Found {len(subdirs)} subdirectories in {DATA_DIR}")

for subdir in tqdm(subdirs, desc="Processing Cohorts"):
    cohort_name = os.path.basename(subdir)
    
    # Determine Group
    assigned_group = None
    for group_name, cohort_list in COHORT_GROUPS.items():
        if cohort_name in cohort_list:
            assigned_group = group_name
            break
    
    if not assigned_group:
        print(f"[WARNING] Cohort '{cohort_name}' not in defined groups. Skipping.")
        continue

    # Find Files
    prot_files = glob.glob(os.path.join(subdir, "protein_matrix_*.txt"))
    rna_files = glob.glob(os.path.join(subdir, "RNA_matrix_*.txt"))
    
    if prot_files and rna_files:
        try:
            df_p = pd.read_csv(prot_files[0], sep="\t", index_col=0)
            df_r = pd.read_csv(rna_files[0], sep="\t", index_col=0)
            
            # Align Samples
            common_samples = df_p.columns.intersection(df_r.columns)
            
            if len(common_samples) > 0:
                df_p = df_p[common_samples]
                df_r = df_r[common_samples]

                # Rename Samples
                new_names = [f"{cohort_name.upper()}_{i+1}" for i in range(len(common_samples))]
                df_p.columns = new_names
                df_r.columns = new_names
                
                # Store
                group_data[assigned_group]["prot"].append(df_p)
                group_data[assigned_group]["rna"].append(df_r)
        except Exception as e:
            print(f"  -> ERROR in {cohort_name}: {e}")

Found 20 subdirectories in ../data/Collection_A


Processing Cohorts: 100%|██████████| 20/20 [00:23<00:00,  1.16s/it]


## 2. Concatenate, Filter, and Save per Group

In [6]:
from src.utils_val import filter_nans

for group_name, dfs in group_data.items():
    print(f"\n=== Processing Group: {group_name} ===")
    
    # Concatenate
    full_protein = pd.concat(dfs["prot"], axis=1, join='outer', sort=True) if dfs["prot"] else pd.DataFrame()
    full_rna = pd.concat(dfs["rna"], axis=1, join='outer', sort=True) if dfs["rna"] else pd.DataFrame()

    # --- MAPPING RNA IDs to SYMBOLS ---
    MAPPING_FILE = os.path.abspath("../data/Collection_A/brain/rows_metadata.csv")
    if os.path.exists(MAPPING_FILE) and not full_rna.empty:
        print(" Mapping RNA Ensembl IDs to Gene Symbols...")
        try:
            map_df = pd.read_csv(MAPPING_FILE)
            map_df = map_df.dropna(subset=['ensembl_gene_id', 'gene_symbol'])
            id2sym = dict(zip(map_df['ensembl_gene_id'], map_df['gene_symbol']))
            
            # Map index
            full_rna.index = full_rna.index.map(lambda x: id2sym.get(x, x))
            
            # Handle duplicates (averaging)
            if full_rna.index.duplicated().any():
                print(f"   -> Found {full_rna.index.duplicated().sum()} duplicate symbols. Averaging...")
                full_rna = full_rna.groupby(level=0).mean()
                
            print("   -> Mapping complete.")
        except Exception as e:
            print(f"   -> Mapping Error: {e}")
    # ----------------------------------
    
    
    if full_protein.empty: 
        print(" [WARNING] No Protein data found.")
        continue
        
    print(f" Initial Shapes -> Protein: {full_protein.shape}, RNA: {full_rna.shape}")

    # Filter
    print(" Filtering Protein...")
    full_protein = filter_nans(full_protein)
    print(" Filtering RNA...")
    full_rna = filter_nans(full_rna)
    
    # Transpose (Samples x Features)
    integrated_protein = full_protein.T
    integrated_rna = full_rna.T
    
    # Remove duplicates
    integrated_protein = integrated_protein.loc[~integrated_protein.index.duplicated(keep='first')]
    integrated_rna = integrated_rna.loc[~integrated_rna.index.duplicated(keep='first')]
    
    print(f" Final Shapes   -> Protein: {integrated_protein.shape}, RNA: {integrated_rna.shape}")

    # --- VIEW HEADS ---
    print(f"Top 5 rows of {group_name} Protein:")
    display(integrated_protein.head())
    print(f"Top 5 rows of {group_name} RNA:")
    display(integrated_rna.head())
    # ------------------
    

    # Save
    p_out = os.path.join(OUTPUT_DIR, f"Collection_A_{group_name}_Protein.csv")
    r_out = os.path.join(OUTPUT_DIR, f"Collection_A_{group_name}_RNA.csv")
    
    integrated_protein.to_csv(p_out)
    integrated_rna.to_csv(r_out)
    print(f" Saved to {p_out} and {r_out}")


=== Processing Group: TMT ===
 Mapping RNA Ensembl IDs to Gene Symbols...
   -> Found 714 duplicate symbols. Averaging...
   -> Mapping complete.
 Initial Shapes -> Protein: (11259, 1924), RNA: (32840, 1924)
 Filtering Protein...
   -> Dropped 4374 Features & 0 Samples
 Filtering RNA...
   -> Dropped 17839 Features & 0 Samples
 Final Shapes   -> Protein: (1924, 6885), RNA: (1924, 15001)
Top 5 rows of TMT Protein:


ensembl_gene_id,ENSG00000000003,ENSG00000000419,ENSG00000000457,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001497,ENSG00000001561,ENSG00000001629,ENSG00000001630,...,ENSG00000278259,ENSG00000278535,ENSG00000278540,ENSG00000278615,ENSG00000278619,ENSG00000278845,ENSG00000281887,ENSG00000283900,ENSG00000284194,ENSG00000284770
BRAIN_2021_1,-0.057,0.054,NaN,0.812,-0.243,0.185,-0.265,0.772,-0.073,-0.673,...,NaN,-0.077,0.177,-0.385,-0.030,0.454,-0.661,-0.032,-0.107,-0.017
BRAIN_2021_2,0.032,0.287,NaN,-0.148,-0.060,0.613,0.062,-1.849,0.016,-0.445,...,NaN,0.159,0.048,0.246,-0.510,0.117,-0.347,-0.083,-0.131,0.205
BRAIN_2021_3,-1.651,-0.424,NaN,0.153,-0.536,0.798,-0.485,-0.054,-0.061,-0.218,...,NaN,0.635,-0.332,-0.047,-0.641,-0.184,0.024,0.266,-0.403,-0.343
BRAIN_2021_4,-0.219,0.568,NaN,-0.700,-0.639,0.368,0.092,-0.751,-0.267,-0.329,...,NaN,0.873,-0.184,0.160,1.006,0.933,-0.158,-0.738,-0.010,0.537
BRAIN_2021_5,0.233,-0.014,NaN,-0.366,0.288,-0.262,-0.053,-0.698,0.081,-0.302,...,NaN,-0.397,-0.328,0.041,0.110,0.224,0.019,-0.183,-0.155,-0.259


Top 5 rows of TMT RNA:


,A1BG,A1CF,A2LD1,A2M,A2ML1,A4GALT,AAAS,AACS,AADAT,AAGAB,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
BRAIN_2021_1,11.942,NaN,13.391,21.515,14.359,13.935,17.915,15.391,16.498,16.998,...,15.307,16.746,16.046,16.371,17.805,NaN,18.028,17.907,17.322,17.947
BRAIN_2021_2,13.370,NaN,14.481,20.523,15.195,12.702,17.412,14.507,15.787,17.885,...,16.008,16.458,16.514,16.285,17.897,NaN,18.140,19.269,17.802,17.957
BRAIN_2021_3,13.056,NaN,15.181,22.874,13.440,15.041,17.377,14.910,16.085,17.882,...,16.538,17.067,15.761,15.482,17.392,NaN,18.172,18.528,17.626,17.746
BRAIN_2021_4,14.537,NaN,15.114,22.091,14.503,14.747,17.714,15.304,17.188,18.098,...,15.595,16.140,16.180,16.132,17.742,NaN,18.090,18.644,17.952,17.941
BRAIN_2021_5,12.671,NaN,13.666,21.372,14.370,15.341,17.604,15.414,16.392,17.923,...,15.453,16.192,15.955,15.619,17.543,NaN,18.019,19.882,17.800,17.609


 Saved to ../data/processed\Collection_A_TMT_Protein.csv and ../data/processed\Collection_A_TMT_RNA.csv

=== Processing Group: iTRAQ ===
 Mapping RNA Ensembl IDs to Gene Symbols...
   -> Found 5 duplicate symbols. Averaging...
   -> Mapping complete.
 Initial Shapes -> Protein: (7569, 276), RNA: (18854, 276)
 Filtering Protein...
   -> Dropped 2575 Features & 0 Samples
 Filtering RNA...
   -> Dropped 4333 Features & 0 Samples
 Final Shapes   -> Protein: (276, 4994), RNA: (276, 14521)
Top 5 rows of iTRAQ Protein:


ensembl_gene_id,ENSG00000000419,ENSG00000000457,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001497,ENSG00000001561,ENSG00000001630,ENSG00000002549,ENSG00000002586,...,ENSG00000277656,ENSG00000277791,ENSG00000277972,ENSG00000278053,ENSG00000278535,ENSG00000278540,ENSG00000278637,ENSG00000278845,ENSG00000284194,ENSG00000284770
BREAST_2016_1,-0.081231,-0.419796,0.176869,-0.713455,-0.323662,0.523479,-0.388918,-1.139969,0.301661,-0.480420,...,-0.819435,0.001128,-0.244252,0.330605,-0.099145,-0.403817,0.139550,0.122242,0.249012,-0.159435
BREAST_2016_2,0.092798,-0.568546,0.094908,-0.269536,-0.386053,0.296988,-0.003414,0.130995,-0.017857,-0.427706,...,-0.821203,0.012873,-0.523509,-0.036846,-0.142420,-0.399615,0.318469,-0.105221,-0.168253,0.552960
BREAST_2016_3,0.076613,-0.161095,0.129747,-0.569031,-0.018495,0.177341,-0.199855,-0.068548,0.359309,-0.379742,...,-0.251332,-0.412622,0.019530,-0.065053,-0.467625,-0.223954,0.538208,0.009289,0.519813,-0.078055
BREAST_2016_4,0.191850,-0.081310,0.293496,-0.302934,0.441546,-0.523233,0.043726,-0.049687,0.821687,-0.695864,...,0.308959,0.300451,-0.184601,-0.563895,0.327181,-0.017601,-0.431829,-0.097583,-0.550864,-0.071434
BREAST_2016_5,-0.175768,-0.442659,-0.072096,0.333302,-0.154080,0.046884,-0.819251,-0.271550,-0.096454,0.491801,...,0.711888,-0.084011,-0.578629,0.043491,-0.569962,0.015199,-0.277400,-0.391153,-0.077394,-0.260433


... (3 outputs left out)

,ENSG00000000419,ENSG00000000971,ENSG00000001084,ENSG00000001561,ENSG00000002549,ENSG00000002586,ENSG00000002834,ENSG00000003056,ENSG00000004059,ENSG00000004142,...,ENSG00000263001,ENSG00000264364,ENSG00000265241,ENSG00000265681,ENSG00000265808,ENSG00000270647,ENSG00000273749,ENSG00000277443,ENSG00000277791,ENSG00000278637
BRAIN_1,7.778,7.033,8.947,7.431,9.103,7.580,9.386,NaN,9.401,7.905,...,7.694,10.007,7.783,9.033,9.028,7.849,NaN,10.055,8.916,NaN
BRAIN_2,7.739,7.033,9.000,7.416,9.385,7.964,9.079,NaN,9.273,7.993,...,8.890,9.930,8.400,9.253,9.225,8.104,NaN,10.296,8.951,NaN
BRAIN_3,7.977,7.033,9.212,8.147,9.464,6.974,9.294,NaN,9.326,6.948,...,7.981,9.906,7.793,9.180,9.201,7.942,NaN,9.911,8.981,NaN
BRAIN_4,7.889,7.033,9.161,7.811,9.439,7.773,9.325,NaN,9.352,6.948,...,7.528,9.768,7.557,9.087,9.254,7.938,NaN,9.975,9.055,NaN
BRAIN_5,8.142,7.033,9.363,7.926,9.405,7.949,9.270,NaN,9.469,7.891,...,6.577,9.860,7.777,8.746,9.187,8.025,NaN,9.799,9.059,NaN


Top 5 rows of LabelFree RNA:


,A1BG,A1CF,A2LD1,A2M,A4GALT,AAAS,AACS,AADAT,AAGAB,AAK1,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11B,ZYX,ZZEF1,ZZZ3
BRAIN_1,12.893,NaN,0.826,21.054,NaN,4.548,2.540,3.030,10.040,1.686,...,2.241,2.650,2.489,0.805,2.664,0.600,11.526,11.135,1.012,1.781
BRAIN_2,8.307,NaN,0.939,20.753,NaN,9.086,1.035,0.474,10.601,2.372,...,3.313,2.422,0.679,1.019,2.700,1.964,10.713,35.617,2.349,1.895
BRAIN_3,15.115,NaN,0.770,18.468,NaN,3.590,2.796,2.194,8.735,1.446,...,1.143,1.566,1.440,0.422,0.939,0.278,6.985,8.945,0.497,1.029
BRAIN_4,2.053,NaN,0.751,35.283,NaN,4.274,2.869,1.592,17.248,5.144,...,3.184,1.881,3.975,0.663,3.019,0.717,15.983,10.402,2.042,2.123
BRAIN_5,6.533,NaN,1.110,29.215,NaN,5.186,1.586,2.978,10.827,1.595,...,2.893,1.973,2.196,0.839,2.132,0.863,6.988,6.625,1.148,1.810


 Saved to ../data/processed\Collection_A_LabelFree_Protein.csv and ../data/processed\Collection_A_LabelFree_RNA.csv
